# B2-024 — Book 1 Bridge Diagnostic

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C3-gradient-descent`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C10-cnn-transfer`, `aio-ml:C11-competition-craft`, `aio-ml:C8-neural-training`, `aio-ml:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C3-gradient-descent](../../../../aio-ml/units/08-C3-gradient-descent/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C10-cnn-transfer](../../../../aio-ml/units/17-C10-cnn-transfer/lesson.ipynb), [aio-ml:C11-competition-craft](../../../../aio-ml/units/18-C11-competition-craft/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [aio-ml:C12-classical-models](../../../../aio-ml/units/19-C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

This 30-minute diagnostic checks the Book 1 tools that B2-024 imports without re-teaching.
Answer both checkpoints in each section before reading its collected answers.
If a section is not fluent, follow the named remediation link before Session 1.
`B2-023-generative-models-diffusion` is a schedule-order predecessor only: no Book 2 concept is required here, and Session 1's forward-only topic merely refers back to it.
Mixed precision, checkpointing with RNG state, the percentile bootstrap, pseudo-labeling, Tikhonov regularization of a grid operator, and permutation-invariant losses are **not** assumed; Sessions 1–6 teach them.

## 1. Arrays, broadcasting, and seeds

Every computation in this unit is batched: rows are examples.
Broadcasting (`aio-ml:broadcasting`) aligns trailing dimensions, so `(N, 1, 2) - (1, S, 2)` gives `(N, S, 2)`, which is how Session 5 computes all source-to-sensor offsets at once.
Tensors behave like `aio-ml:numpy-arrays` with a device and an autograd flag.
`aio-ml:random-seeding` makes runs repeatable; this unit additionally restores generator *state* to resume runs exactly (Session 1) and uses dedicated seeded `torch.Generator` objects for bootstraps (Session 3).
Remediation: [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb).

**Checkpoint 1A.** What is the shape of `a[:, None, :] - b[None, :, :]` for `a` of shape `(200, 2)` and `b` of shape `(8, 2)`?

**Checkpoint 1B.** Two runs seed `torch.manual_seed(0)` but one draws an extra random number before training starts. Do their mini-batches match?

**Collected answers.** `(200, 8, 2)`. No: the extra draw advances the global generator, so every later draw (initialization, batch indices) differs.

## 2. Matrices, rank, and linear least squares

`A @ u` with `A` of shape `(8, 81)` and `u` of shape `(81,)` gives `(8,)` (`aio-ml:matrix-multiplication`).
A square matrix is invertible exactly when it has full rank (`aio-ml:invertibility-via-rank`); `rank(AᵀA) = rank(A)`.
Linear regression (`aio-ml:linear-regression`) minimizes the squared error (`aio-ml:mse-loss`) and solves the normal equations; ridge regression (`aio-ml:l2-regularization`) adds $\lambda\lVert w\rVert^2$, which Session 5 reuses as Tikhonov regularization.
Remediation: [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb) for rank and products; [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb) for least squares and ridge.

**Checkpoint 2A.** $A$ is $8\times81$ with rank 8. What is the rank of $A^\top A$, and is it invertible?

**Checkpoint 2B.** Write the ridge-regression objective for design matrix $X$, targets $y$, and coefficients $w$.

**Collected answers.** Rank 8, so the $81\times81$ matrix $A^\top A$ is singular (not invertible). $\lVert Xw-y\rVert^2+\lambda\lVert w\rVert^2$ with $\lambda>0$.

## 3. Gradients and gradient descent

The gradient (`aio-ml:gradient`) collects partial derivatives; for a sum of squares, $\nabla_w\lVert Xw-y\rVert^2=2X^\top(Xw-y)$ (`aio-ml:sum-of-squares-gradients`).
Gradient descent (`aio-ml:gradient-descent`) steps against the gradient with a learning rate (`aio-ml:learning-rate`); stochastic gradient descent (`aio-ml:stochastic-gd`) uses a random mini-batch per step, which is why Session 1's checkpoints must restore the generator that draws the batches.
Remediation: [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb) for gradients; [aio-ml:C3-gradient-descent](../../../../aio-ml/units/08-C3-gradient-descent/lesson.ipynb) for descent, learning rates, and SGD.

**Checkpoint 3A.** Give $\nabla_u\big(\lVert Au-y\rVert^2+\lambda\lVert u\rVert^2\big)$.

**Checkpoint 3B.** What changes between full-batch gradient descent and SGD, and what randomness does SGD introduce?

**Collected answers.** $2A^\top(Au-y)+2\lambda u$. SGD estimates the gradient from a random mini-batch each step instead of all rows, so the sequence of sampled batches (drawn from a random generator) becomes part of the run.

## 4. Expectation, variance, covariance, and the Gaussian

`aio-ml:expectation` is linear; `aio-ml:variance` is $E[(X-EX)^2]$; for independent variables (`aio-ml:independence`) the variance of a sum is the sum of variances (`aio-ml:variance-of-sums`), and in general covariance terms (`aio-ml:covariance`) are added.
`aio-ml:gaussian-distribution` describes the sensor noise and the shape of each mixture bump; `aio-ml:sampling-simulation` is how all of this unit's data are produced, from seeded generators.
Session 3 uses these facts to derive $\operatorname{Var}(\bar X)=\sigma^2/n$ and the variance of a paired difference.
Remediation: [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb).

**Checkpoint 4A.** $X$ and $Y$ are independent with variances 4 and 9. What is $\operatorname{Var}(X-Y)$?

**Checkpoint 4B.** If $\operatorname{Cov}(X,Y)=3$ instead, what is $\operatorname{Var}(X-Y)$?

**Collected answers.** $4+9=13$ (independence removes the covariance term, and the minus sign does not change a variance). $4+9-2\cdot3=7$.

## 5. Protocols, metrics, and competition craft

`aio-ml:train-test-split` and `aio-ml:hidden-test-protocol` separate fitting, selection, and final scoring; `aio-ml:overfitting` is what a held-out set detects.
`aio-ml:accuracy-precision-recall` and `aio-ml:class-imbalance` matter when pseudo-labels skew the classes (Session 4).
Book 1's competition craft taught `aio-ml:metric-driven-iteration` (change one thing, measure on validation, decide), `aio-ml:writeup-quality` (a reader can reproduce your choices), `aio-ml:colab-coding-submission` (the hosted-notebook submission routine), and `aio-ml:cpu-and-gpu-round-boundary` (Round 1 is CPU-only; Round 2 may grant an accelerator).
Sessions 1–3 extend each of these to budgeted, GPU-aware, open-ended work.
Remediation: [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb) for splits, overfitting, and metrics; [aio-ml:C11-competition-craft](../../../../aio-ml/units/18-C11-competition-craft/lesson.ipynb) for the hidden-test protocol, iteration, writeups, Colab submission, and the round boundary.

**Checkpoint 5A.** Which split chooses hyperparameters, and how many times is the locked test set scored?

**Checkpoint 5B.** On a pool that is 70% class 0, a classifier predicts class 0 for everything. Give its accuracy and its recall on class 1.

**Collected answers.** The validation split; the test set is scored once, after all fitting and selection. Accuracy $0.7$; recall on class 1 is $0$.

## 6. Neural networks, CNNs, and the PyTorch training loop

An MLP (`aio-ml:mlp-architecture`) alternates linear layers with `aio-ml:relu-activation`; a classifier ends in logits, and `aio-ml:softmax` with `aio-ml:cross-entropy-loss` (`F.cross_entropy`) trains it.
`aio-ml:torch-tensors`, `aio-ml:nn-module`, and `aio-ml:requires-grad` are the PyTorch building blocks; `aio-ml:parameter-counting` is the basis of Session 1's memory arithmetic.
`aio-ml:convolution` and `aio-ml:cnn-training` give the small CNN used for the shapes tasks.
`aio-ml:torch-optimizers` (Adam), `aio-ml:autograd-training` (zero_grad → forward → loss → backward → step), and `aio-ml:trained-mlp` (a trained network evaluated on held-out data) are used in every training practice.
Remediation: [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb); [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb); [aio-ml:C10-cnn-transfer](../../../../aio-ml/units/17-C10-cnn-transfer/lesson.ipynb); [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb).

**Checkpoint 6A.** How many parameters does `nn.Conv2d(1, 8, 3)` have, and what spatial size does it produce from an $8\times8$ input with no padding?

**Checkpoint 6B.** What goes wrong if `optimizer.zero_grad()` is omitted from the loop?

**Collected answers.** $8\cdot(1\cdot3\cdot3)+8=80$; output $6\times6$ per channel. Gradients accumulate across steps, so each update uses the sum of all past gradients.

## 7. k-means and Lloyd's algorithm

`aio-ml:k-means` partitions points into $k$ clusters by minimizing the within-cluster sum of squared distances; `aio-ml:lloyd-algorithm` alternates assigning each point to its nearest centroid and moving each centroid to the mean of its points.
Session 4 runs it on frozen CNN features, with a stated initialization, iteration count, and tie rules, to name clusters with a few labels.
Remediation: [aio-ml:C12-classical-models](../../../../aio-ml/units/19-C12-classical-models/lesson.ipynb).

**Checkpoint 7A.** Points $0, 1, 9, 10$ on a line, centroids initialized at $0$ and $10$. Give the centroids after one Lloyd iteration.

**Checkpoint 7B.** What should happen to a centroid with no assigned points, under this unit's rule?

**Collected answers.** Assignments $\{0,1\}$ and $\{9,10\}$; centroids $0.5$ and $9.5$. It stays where it is (this unit's stated rule).

## 8. Readiness and explicit remediation

Proceed to Session 1 only if every checkpoint above was fluent; otherwise use the link for the exact gap.
The unit's own new tools are taught from scratch in Sessions 1–6.
[B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb) precedes this unit in the schedule only.
The cell below checks that the unit's data module loads and that the CPU path works.

**Checkpoint 8A.** Which link do you follow if you cannot derive the gradient of a sum of squares?

**Checkpoint 8B.** Do you need a GPU to complete any required part of this unit?

**Collected answers.** [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb) (then C2 for least squares). No: every required lesson and practice runs on a CPU; accelerator extensions are optional.

In [ ]:
import importlib.util
from pathlib import Path
import torch

# Boilerplate: find and load this unit's data module
candidates = (Path("../data/capstone_data.py"), Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
              Path("aio-dl/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"))
path = next(p for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location("capstone_data", path)
capstone_data = importlib.util.module_from_spec(spec)
spec.loader.exec_module(capstone_data)

# torch.cuda.is_available() is True when a CUDA GPU is usable;
# torch.device names where tensors live ("cuda" or "cpu")
# torch.linalg.matrix_rank(M) returns the rank of M (number of nonzero
# singular values) as a 0-d tensor; torch.ones, torch.eye, .item as usual
# torch.zeros: tensor of zeros
readiness = {
    "broadcast shape": (torch.zeros(200, 1, 2) - torch.zeros(1, 8, 2)).shape == (200, 8, 2),
    "rank of A^T A": torch.linalg.matrix_rank(torch.ones(8, 81) + torch.eye(8, 81)).item() <= 8,
    "data module tasks": capstone_data.TASKS == ("shapes_supervised", "shapes_ssl", "inverse", "mixture"),
    "split sizes": [len(capstone_data.TRAIN_IDS[t]) for t in capstone_data.TASKS] == [600, 30, 800, 800],
    "CPU path": torch.device("cuda" if torch.cuda.is_available() else "cpu").type in ("cpu", "cuda"),
}
print(readiness)
print("ready for B2-024" if all(readiness.values()) else "follow the remediation links first")